# v41 — Windows / RTX 4060 8 GB

Сначала выполнить `setup_windows.ps1` и выбрать kernel **Car ReID RTX 4060**. Потом Run All.
Три независимые очереди, последовательно на одной GPU: TransReID, NiVe, DINOv2/R50-IBN.
12 TransReID-конфигураций (10 → 30 проходов, лучшие 4 → 90); 9 NiVe + 9 target-extra + target-only; 10 frozen внешних представлений и до 14 дообучений лучших источников.
Не ограничено одной ночью: длительность зависит от устройства. Прогресс и checkpoints сохраняются; повторный Run All продолжает тот же run.
Никаких данных из теста/outer в обучении, никакого автоматического promotion. Контроль C_primary, **не метрика релизного v25**.
NiVe1303 v1: https://data.mendeley.com/datasets/42wv2svztx/1 — train-фотографии уже входят в отдельный входной архив; в Git их нет.


In [ ]:
import os, sys, subprocess, json
from pathlib import Path
os.environ['ORT_DISABLE_TELEMETRY'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['XFORMERS_DISABLED'] = '1'
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'training/research_gpu.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Открой notebook внутри клона Car-classification-MSK / fine-tuning')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
INPUTS = ROOT / 'research_transfer/v41_inputs'
print('Python:', sys.executable)
print('Repo:', ROOT)
print('Inputs:', INPUTS)
if not (INPUTS / 'inputs.json').is_file():
    raise FileNotFoundError('Распакуй v41_inputs.zip в research_transfer/v41_inputs. См. README варианта.')
RUN_NAME = 'rtx4060_v1'


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_parallel_research.py'], cwd=ROOT, env=os.environ.copy(), check=True)
subprocess.run([sys.executable, '-c', "import torch; assert torch.cuda.is_available(), 'CUDA недоступна, CPU fallback запрещён'; p=torch.cuda.get_device_properties(0); print(p.name, round(p.total_memory/2**30,2), 'GiB'); print('Torch:',torch.__version__,'CUDA:',torch.version.cuda)"], check=True, env=os.environ.copy())

In [ ]:
from training.research_gpu import run
summary = run(INPUTS, RUN_NAME, device='cuda')


In [ ]:
print('Статус:', summary['status'])
for item in summary['leaderboard'][:10]:
    print(item['trial'], 'rung', item['rung'], 'G2:', round(item['means']['G2'],6), 'delta:', round(item['delta'],6))
print('Передать обратно ОБА архива:')
for suffix in ('analysis', 'full_results'):
    print(ROOT / f'OSNet-AIN-x1.0/variant_41_windows_queues/{RUN_NAME}_{suffix}.zip')
